# CIFAR-10 AVG v2 — Final Testing

Load the saved AVG v2 CNN and SVM, then report official CIFAR-10 test accuracy. This notebook does not fit or select models.


In [ ]:
import os
import random
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline

SEED = 30092026

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

import tensorflow as tf

# Hindari TensorFlow langsung mengalokasikan hampir seluruh VRAM.
gpus = tf.config.list_physical_devices("GPU")
for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        pass

print("GPU:", gpus)

tf.random.set_seed(SEED)

from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Model, load_model

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

## 1. Load Official CIFAR-10 Test Set

In [ ]:
_, (X_test, y_test) = cifar10.load_data()

y_test = y_test.flatten()

labels = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]

classes_name = [
    "Airplane", "Automobile", "Bird", "Cat", "Deer",
    "Dog", "Frog", "Horse", "Ship", "Truck"
]

print("Original RGB X_test:", X_test.shape)
print("y_test             :", y_test.shape)

In [ ]:
classes, counts = np.unique(
    y_test,
    return_counts=True
)

plt.figure(figsize=(8, 5))
plt.barh(
    classes_name,
    counts
)
plt.title(
    "Distribusi Kelas CIFAR-10 — Test Set"
)
plt.xlabel("Jumlah gambar")
plt.show()

## 2. Preprocessing Grayscale AVG

Preprocessing harus identik dengan AVG Training:

```text
AVG = (R + G + B) / 3
```

kemudian normalisasi ke `[0, 1]`.

In [ ]:
X_test = np.mean(
    X_test.astype("float32"),
    axis=-1,
    keepdims=True
) / 255.0

y_cat_test = to_categorical(
    y_test,
    10
)

print("AVG X_test :", X_test.shape)
print("y_cat_test :", y_cat_test.shape)

assert X_test.shape == (10000, 32, 32, 1)

## 3. Load Grayscale AVG CNN dan SVM

In [ ]:
CNN_MODEL_PATH = "cifar10_avg_cnn_v2.keras"
SVM_MODEL_PATH = "cifar10_avg_svm_v2.joblib"

if not os.path.exists(CNN_MODEL_PATH):
    raise FileNotFoundError(
        f"{CNN_MODEL_PATH} tidak ditemukan. "
        "Jalankan notebook training terlebih dahulu."
    )

if not os.path.exists(SVM_MODEL_PATH):
    raise FileNotFoundError(
        f"{SVM_MODEL_PATH} tidak ditemukan. "
        "Jalankan notebook training terlebih dahulu."
    )

# Bersihkan session TensorFlow lama agar penggunaan VRAM lebih aman.
tf.keras.backend.clear_session()

# CNN hanya digunakan untuk inference / feature extraction.
cnn_model = load_model(
    CNN_MODEL_PATH,
    compile=False
)

# SVM Pipeline berisi StandardScaler + trained RBF-SVM.
svm_model = joblib.load(
    SVM_MODEL_PATH
)

print("CNN berhasil dimuat dari:")
print(CNN_MODEL_PATH)

print("\nSVM berhasil dimuat dari:")
print(SVM_MODEL_PATH)

print("\nCNN summary:")
cnn_model.summary()

print("\nSVM pipeline:")
print(svm_model)

# 4. Feature Map Extraction pada AVG Test Image

Menampilkan feature maps dari satu unseen Grayscale AVG test image.

In [ ]:
FEATURE_MAP_LAYERS = [
    "stem_relu",
    "stage2_block2_out",
    "stage3_block2_out",
    "feature_map"
]

feature_map_model = Model(
    inputs=cnn_model.input,
    outputs=[
        cnn_model.get_layer(name).output
        for name in FEATURE_MAP_LAYERS
    ],
    name="avg_feature_map_extractor"
)

sample_index = 100
sample_image = X_test[
    sample_index:sample_index + 1
]

feature_maps = feature_map_model.predict(
    sample_image,
    verbose=0
)

plt.figure(figsize=(3, 3))
plt.imshow(sample_image[0, :, :, 0], cmap="gray", vmin=0.0, vmax=1.0)
plt.title(
    f"Test image: "
    f"{labels[y_test[sample_index]]}"
)
plt.axis("off")
plt.show()

for layer_name, fmap in zip(
    FEATURE_MAP_LAYERS,
    feature_maps
):
    n_show = min(
        12,
        fmap.shape[-1]
    )

    print(
        f"{layer_name}: {fmap.shape}"
    )

    plt.figure(figsize=(12, 4))

    for i in range(n_show):
        plt.subplot(
            3,
            4,
            i + 1
        )
        plt.imshow(
            fmap[0, :, :, i],
            cmap="viridis"
        )
        plt.title(
            f"Map {i + 1}"
        )
        plt.axis("off")

    plt.suptitle(
        f"Feature Maps — {layer_name}"
    )
    plt.tight_layout()
    plt.show()

## 5. Baseline: Evaluasi AVG CNN Softmax

CNN Softmax digunakan sebagai pembanding.

Hasil utama eksperimen tetap **AVG CNN feature extraction + SVM**.

In [ ]:
cnn_probability = cnn_model.predict(
    X_test,
    batch_size=128,
    verbose=1
)

cnn_prediction = np.argmax(
    cnn_probability,
    axis=1
)

cnn_accuracy = accuracy_score(
    y_test,
    cnn_prediction
)

print(
    f"CNN test accuracy: "
    f"{cnn_accuracy * 100:.2f}%"
)

# 6. Extract AVG CNN Features untuk Seluruh Test Set

SVM menerima output layer `svm_features`, yaitu 512-dimensional vector dari Grayscale AVG CNN.

In [ ]:
feature_extractor = Model(
    inputs=cnn_model.input,
    outputs=cnn_model.get_layer(
        "svm_features"
    ).output,
    name="avg_cnn_feature_extractor"
)

X_test_features = feature_extractor.predict(
    X_test,
    batch_size=128,
    verbose=1
)

print(
    "Original test shape:",
    X_test.shape
)

print(
    "Extracted feature shape:",
    X_test_features.shape
)

# 7. Klasifikasi RBF-SVM pada AVG CNN Features

In [ ]:
svm_prediction = svm_model.predict(X_test_features)
svm_accuracy = accuracy_score(y_test, svm_prediction)
print("AVG CNN softmax test accuracy:", f"{cnn_accuracy:.4%}")
print("AVG CNN features + SVM test accuracy:", f"{svm_accuracy:.4%}")
print("Optional individual >95% milestone:", "PASS" if svm_accuracy > 0.95 else "NOT YET")

## 8. Classification Report SVM

In [ ]:
print(
    classification_report(
        y_test,
        svm_prediction,
        target_names=labels,
        digits=4
    )
)

## 9. Perbandingan AVG CNN vs AVG CNN Features + SVM

In [ ]:
comparison = pd.DataFrame({
    "Classifier": [
        "CNN Softmax",
        "SVM on CNN Features"
    ],
    "Test Accuracy": [
        cnn_accuracy,
        svm_accuracy
    ]
})

display(comparison)

## 10. Confusion Matrix — SVM

In [ ]:
cm = confusion_matrix(
    y_test,
    svm_prediction
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=labels
)

fig, ax = plt.subplots(
    figsize=(10, 10)
)

disp.plot(
    ax=ax,
    xticks_rotation="vertical",
    cmap="summer"
)

plt.title(
    "AVG CNN Feature Extraction + SVM "
    "— Confusion Matrix"
)

plt.show()

## 11. Beberapa Hasil Prediksi

In [ ]:
num_rows = 3
num_cols = 5
num_images = (
    num_rows * num_cols
)

plt.figure(figsize=(15, 9))

for i in range(num_images):
    plt.subplot(
        num_rows,
        num_cols,
        i + 1
    )

    plt.imshow(
        X_test[i, :, :, 0],
        cmap="gray",
        vmin=0.0,
        vmax=1.0
    )

    predicted = int(
        svm_prediction[i]
    )

    actual = int(
        y_test[i]
    )

    plt.title(
        f"Pred: {labels[predicted]}\n"
        f"True: {labels[actual]}"
    )

    plt.axis("off")

plt.tight_layout()
plt.show()

# AVG v2 official test complete

The reported scores use saved AVG v2 CNN and SVM artifacts. No model fitting or parameter selection occurs in this notebook.
